This notebook demonstrates basic FCS file manipulation in Python using the [`flowkit`](https://flowkit.readthedocs.io/en/latest/index.html) package.

## Installation

First, we have to make sure that we have the correct packages installed

In [ ]:
# Install Flowkit
%pip install flowkit
# Install UMAP
%pip install umap-learn
# Install HDBSCAN
%pip install hdbscan

## Load the Packages

Now that we've installed FlowKit, we can import it (and a few other plotting functions) into Python.

In [ ]:
# Import FlowKit itself:
import flowkit
import bokeh
from bokeh.plotting import show
import matplotlib.pyplot as plt
import umap
import hdbscan

In [ ]:
# These commands are some magic to allow this notebook to nicely capture and show the plot images we're going to generate
bokeh.io.output_notebook()
%matplotlib inline
_ = plt.ioff()

## Obtain Example Data

We're going play with a single FCS file downloaded from [Github](https://github.com/alastair-droop/python-flowkit).

In [ ]:
# Pull an example FCS file from GitHub for this demonstration:
!wget https://github.com/alastair-droop/python-flowkit/raw/main/examples/example.fcs

## Load the FCS Data into Python

Once we've loaded the FlowKit package and downloaded the FCS file itself, we can load it into Python

In [ ]:
# Load the FCS file into Python:
sample = flowkit.Sample("./example.fcs", subsample=10000)

In [ ]:
# Inspect the sample:
sample

In [ ]:
# Pull out the total number of events in the sample:
sample.event_count

In [ ]:
# Inspect the sample metadata:
sample.metadata

In [ ]:
# Inspect the sample channels:
sample.channels

In [ ]:
# Identify the fluorescent channels:
sample.fluoro_indices

In [ ]:
# Identify the time index:
sample.time_index

In [ ]:
# We can see the raw data as a DataFrame:
sample_data = sample.as_dataframe(source="raw")
sample_data

## Plotting Raw Channels

We can plot raw channel data as an initial QC of the data we have received.

**NB**: We will certainly want to perform transformations and gating before we analyse our data, but it is always a good idea to visualise your raw data before proceeding.

In [ ]:
# Plot a histogram of the counts over time:
p = sample.plot_histogram("Time", source="raw")
show(p)

In [ ]:
# Plot FL6-A over time:
p = sample.plot_scatter("Time", "FL6-A", source="raw")
show(p)

As well as plotting data aginst time, we can easily plot x,y data using `plot_scatter`

In [ ]:
# Plot forward scatter by side scatter:
p = sample.plot_scatter("FSC-A", "SSC-A", source="raw")
show(p)

In [ ]:
# Plot FL1-A vs FL10-A:
p = sample.plot_scatter("FL1-A", "FL10-A", source="raw")
show(p)

## Simple Filtering

We can remove negative scatter values using the `filter_negative_scatter()` function

In [ ]:
# Filter out negative scatter:
sample.filter_negative_scatter()
# Pull out the total (remaining) number of events in the sample:
# NB: Remember we had 11319 before filtering negative scatter values, so in this case we've not removed any events.
sample.event_count

## Apply Compensation

The FCS file has a compensation matrics included in its metadata (as a *spillover* matrix).  We can pull this out and apply it to the sample data. Note that in the metadata it is stored in `.csv` format.

In [ ]:
# Get the compensation matrix:
compensation_matrix = sample.metadata["spillover"]
compensation_matrix

In [ ]:
# Apply compensation to the sample:
sample.apply_compensation(compensation_matrix)

Now that we've run the compensation step, we no longer need to use the `raw` data; we can also now use the compensated (`comp`) data.

In [ ]:
# Re-plot FL1-A vs FL10-A, but this time with compensated data:
p = sample.plot_scatter("FL1-A", "FL10-A", source="comp")
show(p)

## Transformations

The data look a bit rubbish, as we're using a linear scale.  Let's transform the data to a more appropriate scale.

There are a variety of builtin transformations we could use, including:

* `LinearTransform`
* `LogTransform`
* `RatioTransform`
* `HyperlogTransform`
* `LogicleTransform`
* `AsinhTransform`
* `WSPLogTransform`
* `WSPBiexTransform`

We're using the `AsinhTransform`, but feel free to experiment.

Each transformation needs some extra data (parameters). In the case of the AsinhTransform transformation. Many GUI-based programs will give you sensible defaults for these, but not here!

The `AsinhTransform` transformation maps raw data onto a [0, 1] axis using the following formula:

$$
y = \frac{\operatorname{asinh}\!\left( \frac{x \, \sinh(M \ln 10)}{T} \right) + A \ln 10}{(M + A) \ln 10 }
$$

The three parameters we therefore need to supply are:

| Parameter | Python name | Description
|:----------|:------------|:-----------
| $T$       | `param_t`   | The top of the scale. This value will become 1.0 after transform
| $M$       | `param_m`   | How many powers of 10 does the positive side of the axis cover
| $A$       | `param_a`   | Extra room below zero, measured in powers of 10

In [ ]:
# Transform using a Logicle transform:
asinh_xform = flowkit.transforms.AsinhTransform(
    param_t=1000,
    param_m=0.5,
    param_a=1.0
)
sample.apply_transform(asinh_xform)

In [ ]:
# Plot FL15 vs FL10 after transform:
p = sample.plot_scatter("FL1-A", "FL10-A", source="xform", subsample=False)
show(p)

## Gating

We can use GateML-formatted gating files to perform gating. We'll use an example file to show this process. As above, we'll pull a file from GitHub to demonstrate this

In [ ]:
# Pull an example FCS file from GitHub for this demonstration:
!wget https://github.com/alastair-droop/python-flowkit/raw/main/examples/example-gating.xml

In [ ]:
# Load the gate from file:
gate = flowkit.parse_gating_xml("./example-gating.xml")

In [ ]:
print(gate.get_gate_hierarchy())

In [ ]:
# Perform gating on the sample:
gated = gate.gate_sample(sample, verbose=True)

In [ ]:
# Check how many cells were included by this gate:
print(gated.get_gate_count("singles"))

In [ ]:
# Extract the elements that were included in a specified gate:
gate_members = gated.get_gate_membership("singles")

In [ ]:
# Plot FL15 vs FL8 after transform:
p = sample.plot_scatter("FL1-A", "FL10-A", source="xform", subsample=False, event_mask=gate_members)
show(p)

## UMAP

Now that we've performed some analyses, let's try performing UMAP on the data and then cluster it with [HDBSCAN](https://hdbscan.readthedocs.io/en/latest/basic_hdbscan.html).

**NB**: We're going to use all the fluorescence channels here (from `sample.fluoro_indices`, but in reality you should use the appropriate ones for your analysis).

In [ ]:
# Extract the subsampled data:
singles_data = sample.get_events(source="xform", event_mask=gate_members)

In [ ]:
singles_data.shape

In [ ]:
# Pull out the transformed samples that passed gating:
fluro_data = singles_data[:, sample.fluoro_indices]

**NB**: Really, we should normalise all of our fluorescence channels to have the same mean and standard deviation, so they are all directly comparable. This is easy to do, but in reality has little impact on the data

In [ ]:
# Normalise all fluorescence channels:
fluro_data = (fluro_data - fluro_data.mean(0)) / fluro_data.std(0)

In [ ]:
# Perform UMAP on transformed counts that passed gating:
res = umap.UMAP(
    n_neighbors = 50,
    min_dist = 0.5
).fit_transform(fluro_data)

In [ ]:
res.shape

In [ ]:
# Use HDBSCAN to cluster these data
res_labels = hdbscan.HDBSCAN(min_cluster_size=50).fit_predict(res)

In [ ]:
res_labels

In [ ]:
# HDBSCAN can fail to cluster some data.  These get a label of -1. Let's remove them
clustered = (res_labels >= 0)

# Plot the non-clustered events in light grey and the clustered ones in colour:
plt.scatter(res[~clustered,0], res[~clustered,1], marker=".", linewidths=0.01, color=(0.95, 0.95, 0.95))
plt.scatter(res[clustered,0], res[clustered,1], marker=".", linewidths=0.01, c=(res_labels[clustered]))
plt.show()